### **Cerebras CS-3: Batch Size Experiment (Llama-2-7B)**

I ran the provided Llama-2-7B training example on the CS-3 appliance with three different global batch sizes (1024, 512, 256) to compare throughput and runtime. All runs were stopped at step 50 for a consistent comparison.

| Batch Size | Loss @ Step 50 | Throughput (Rate) | Global Rate | Estimated Time Remaining |
|------------|----------------|--------------------|--------------|---------------------------|
| **1024**   | 7.67126        | 31.97 samples/s    | 31.97        | ~1 hr 20 min              |
| **512**    | 7.81506        | 33.36 samples/s    | 33.21        | ~39 min                   |
| **256**    | 7.93313        | 32.97 samples/s    | 32.69        | ~20 min                   |

The smaller batch sizes complete faster, since each step processes fewer samples. Throughput stays in a tight range (roughly 32–33 samples/s) across all runs. This is expected on CS-3: the pipeline is built to keep compute steady, so batch size mainly affects how much work each step has to handle, not the throughput itself.

On CS-3, throughput for this model is nearly batch-size invariant. Smaller batches cut the wall-clock time for a fixed number of steps, while larger batches don’t yield higher samples/sec but simply increase total compute per step. In short: batch size affects runtime much more than throughput on this architecture.


### SambaNova Homework: GPT-OSS on Metis vs Sophia

For this homework, I compared inference performance of GPT-OSS deployments on the **Metis** (SN40L / SambaStack) and **Sophia** (vLLM) clusters, using the same set of 50 prompts drawn from a small HuggingFace text dataset (e.g. `cnn_dailymail` validation split). 

Each row summarizes 50 independent requests.

| Cluster | Model                      | Avg Latency / Request | Median Latency | Min Latency | Max Latency |
|---------|---------------------------|------------------------|----------------|-------------|-------------|
| Metis   | `gpt-oss-120b-131072`     | **0.96 s**             | 0.96 s         | 0.68 s      | 1.60 s      |
| Sophia  | `openai/gpt-oss-120b`     | **2.39 s**             | 2.05 s         | 1.42 s      | 10.45 s     |

On Sophia, there is a clear outlier in the first call (~10.45 s), which is likely a cold-start or model warmup. Ignoring that one, typical latencies fall roughly in the **1.9–2.6 s** range.

Overall, Sophia’s average latency is about **2.5× higher** than Metis for this workload, with Metis consistently returning responses in under ~1.2 seconds.
rder of 1–2 seco
For this 50-prompt benchmark:

- **Metis (SN40L / SambaStack)** delivers **sub-second to ~1.6 s** latencies consistently for GPT-OSS, making it well suited for interactive workloads where response time is critical.
- **Sophia (vLLM)** returns similar-quality answers from its GPT-OSS deployment but with typical latencies around **2 seconds per request**, plus occasional cold-start spikes.

In other words, the **model family is similar but the serving stacks differ**: Metis behaves like a tightly integrated inference appliance, while Sophia exposes GPT-OSS through a more general-purpose vLLM setup, which appears to trade some latency for flexibility, throughput, and multi-model support.